# Legal-RAG: Pipeline Ingestion & Retrieval Test
Notebook này chạy bộ Chunker & Embedder 100% On-Premise và tương tác trực tiếp với Local Docker DB.

In [1]:
import sys
import os

# 1. Trỏ vào thư mục gốc Legal-RAG
sys.path.append(os.path.abspath("d:/iCOMM/Legal-RAG"))

# 2. Ghi đè cấu hình Môi trường thành Cục Bộ (Local Docker)
os.environ["QDRANT_URL"] = "http://localhost:6335"
os.environ["QDRANT_API_KEY"] = "" 

# BỔ SUNG DÒNG NÀY ĐỂ ĐỒNG BỘ TÊN COLLECTION
os.environ["QDRANT_COLLECTION"] = "legal_hybrid_rag_docs"

os.environ["NEO4J_URI"] = "bolt://localhost:7687"
os.environ["NEO4J_USERNAME"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "u7aGQYEWeFJD-jyeHB4ATtoAud73PptW35M1RzFlT-0"

# 3. Setup HF Cache
os.environ["HF_DATASETS_CACHE"] = r"D:\huggingface_cache"

print(f"✅ Môi trường gắn kết thành công tới Root Directory.")

✅ Môi trường gắn kết thành công tới Root Directory.


In [2]:
import pandas as pd
from datasets import load_dataset
from backend.utils.document_parser import DocumentParser
from backend.retrieval.chunker.metadata import normalize_doc_key

# 1. Đọc 100 ID từ file CSV được yêu cầu
csv_path = r"D:\iCOMM\Legal-RAG\top_8000_y_te_theo_quyen_luc.csv"
target_ids = set()
if os.path.exists(csv_path):
    df_csv = pd.read_csv(csv_path)
    if "id" in df_csv.columns:
        target_ids = set(df_csv["id"].astype(str).tolist())
        print(f"-> Đã đọc {len(target_ids)} IDs văn bản.")
else:
    print("⚠️ Không tìm thấy file CSV!")

# 2. Tải Dataset Full từ HuggingFace
print("-> Đang tải dataset từ HuggingFace...")
ds_metadata_all = load_dataset("nhn309261/vietnamese-legal-docs", "metadata", split="data")
ds_content_all = load_dataset("nhn309261/vietnamese-legal-docs", "content", split="data")

# 3. Lập chỉ mục O(1) Memory mapping (Tránh bùng nạp RAM)
df_meta = ds_metadata_all.to_pandas()
df_meta["id"] = df_meta["id"].astype(str)
all_meta_records = df_meta.to_dict("records")
print("-> Đang lập chỉ mục O(1)...")
meta_id_to_idx = {row["id"]: idx for idx, row in enumerate(all_meta_records)}
meta_docnum_to_id = {}
for row in all_meta_records:
    doc_num = str(row.get("document_number", ""))
    if doc_num:
        key = normalize_doc_key(doc_num)
        if key:
            meta_docnum_to_id[key] = row["id"]

content_id_to_idx = {str(row["id"]): i for i, row in enumerate(ds_content_all.select_columns(["id"]))}

print(f"✅ Đã tải và lập map {len(all_meta_records)} tài liệu Metadata.")

class DynamicContentLookup(dict):
    def __init__(self, ds_content, content_id_to_idx, ds_meta, meta_docnum_to_id, meta_id_to_idx):
        self.ds_content = ds_content
        self.content_id_to_idx = content_id_to_idx
        self.ds_meta = ds_meta
        self.meta_docnum_to_id = meta_docnum_to_id
        self.meta_id_to_idx = meta_id_to_idx
    def get(self, key, default=None):
        if key in self:
            return super().get(key)
        t_id_str = self.meta_docnum_to_id.get(key)
        if t_id_str:
            idx_cont = self.content_id_to_idx.get(t_id_str)
            if idx_cont is not None:
                text = self.ds_content[idx_cont].get("content", "")
                self[key] = text
                return text
        return default

global_doc_lookup = DynamicContentLookup(ds_content_all, content_id_to_idx, all_meta_records, meta_docnum_to_id, meta_id_to_idx)
meta_by_docnum_lookup = {}


-> Đã đọc 5000 IDs văn bản.
-> Đang tải dataset từ HuggingFace...
-> Đang lập chỉ mục O(1)...
✅ Đã tải và lập map 518255 tài liệu Metadata.


In [3]:
from backend.retrieval.chunker.relations import extract_ontology_relationships_batch
import time
import json

test_doc = '15/2020/NĐ-CP'
test_text = '''Đoạn 1 rác rưởi không có gì.\n
Nghị định 15/2020/NĐ-CP này quy định về xử phạt vi phạm hành chính.\n
Theo quy định tại khoản 2 Điều 3 Nghị định số 15/2020/NĐ-CP, hành vi này bị cấm.\n
Căn cứ Nghị định số 123/2021/NĐ-CP của Chính phủ, sửa đổi bổ sung một số điều của Nghị định 15/2020/NĐ-CP.\n
Nghị định này thay thế Nghị định 174/2013/NĐ-CP.\n
Đây là đoạn rác dài dòng lê thê không chứa từ khóa pháp lý nào cả.'''

print('Bắt đầu test trích xuất (để kiểm tra xem có lấy đoạn thừa hoặc filter sai không)...')
start_time = time.time()
res = extract_ontology_relationships_batch([{'source_doc': test_doc, 'content': test_text}], global_doc_lookup={})
end_time = time.time()

print(f'\nThời gian chạy: {end_time - start_time:.2f} giây')
print('\nKết quả trích xuất:\n')
print(json.dumps(res, ensure_ascii=False, indent=2))


Bắt đầu test trích xuất (để kiểm tra xem có lấy đoạn thừa hoặc filter sai không)...
      [Pha 1] 🚀 Tìm thấy 2 đoạn văn quan trọng. Đang xử lý 1 mẻ (8 đoạn/prompt)...
      [Hoàn tất] ✓ Trích xuất 2 quan hệ từ 1 văn bản.

Thời gian chạy: 30.22 giây

Kết quả trích xuất:

{
  "15/2020/NĐ-CP": [
    {
      "source_doc": "15/2020/NĐ-CP",
      "target_doc": "123/2021/NĐ-CP",
      "relation_phrase": "căn cứ nghị định số 123/2021/nđ-cp",
      "edge_label": "BASED_ON",
      "context": "Căn cứ Nghị định số 123/2021/NĐ-CP của Chính phủ, sửa đổi bổ sung một số điều của Nghị định 15/2020/NĐ-CP.",
      "target_article": "",
      "target_clause": "",
      "target_text": "Căn cứ pháp lý để ban hành hoặc thực hiện"
    },
    {
      "source_doc": "15/2020/NĐ-CP",
      "target_doc": "174/2013/NĐ-CP",
      "relation_phrase": "thay thế nghị định 174/2013/nđ-cp",
      "edge_label": "REPLACES",
      "context": "Căn cứ Nghị định số 123/2021/NĐ-CP của Chính phủ, sửa đổi bổ sung một số điều của N

In [ ]:
from backend.retrieval.chunker.core import AdvancedLegalChunker
from backend.retrieval.chunker.metadata import normalize_doc_key
from backend.retrieval.chunker import relations as rel
chunker = AdvancedLegalChunker()

all_chunks = []
processed_ids = set()
discovered_depth1_ids = set()  # ID văn bản tham chiếu cần nạp thêm
MAX_RECURSIVE_DOCS = 200
BATCH_SIZE = 8

# ═══════════════════════════════════════════════════
# PHA 1: Chunk toàn bộ văn bản GỐC (depth=0) + Phát hiện Target Docs
# ═══════════════════════════════════════════════════
print("═" * 60)
print("PHA 1: Chunking văn bản GỐC (depth=0) + Discovery Target Docs (BATCH 10)")
print("═" * 60)

target_ids_list = list(target_ids)
for i in range(0, len(target_ids_list), BATCH_SIZE):
    batch_ids = target_ids_list[i : i + BATCH_SIZE]
    batch_docs = []
    batch_meta_map = {}
    
    for doc_id in batch_ids:
        if doc_id in processed_ids: continue
        
        idx_content = content_id_to_idx.get(doc_id)
        idx_meta = meta_id_to_idx.get(doc_id)
        if idx_content is None or idx_meta is None:
            processed_ids.add(doc_id)
            continue
        
        content = ds_content_all[idx_content].get("content", "")
        meta = all_meta_records[idx_meta]
        if not content: 
            processed_ids.add(doc_id)
            continue
            
        doc_num = str(meta.get("document_number", ""))
        batch_docs.append({"source_doc": doc_num, "content": content})
        batch_meta_map[doc_num] = {"id": doc_id, "meta": meta, "content": content}

    if not batch_docs: continue
    
    # BƯỚC 1: TRÍCH XUẤT RELATION THEO MẺ 10 (Cell 3 & 4)
    print(f"  [BATCH] Đang trích xuất quan hệ cho mẻ {len(batch_docs)} văn bản...")
    
    # Bọc Try-Catch để đảm bảo pipeline không bao giờ chết
    try:
        rels_map = rel.extract_ontology_relationships_batch(batch_docs, global_doc_lookup=global_doc_lookup)
    except Exception as e:
        print(f"  ⚠️ LỖI BATCH: Bỏ qua trích xuất quan hệ mẻ này do lỗi LLM - {e}")
        rels_map = {} # Dự phòng rỗng để vẫn chunk được văn bản gốc
    
    # BƯỚC 2: CHUNK TỪNG VĂN BẢN VỚI RELATION ĐÃ CÓ
    for doc_info in batch_docs:
        s_doc = doc_info["source_doc"]
        doc_data = batch_meta_map[s_doc]
        doc_id = doc_data["id"]
        meta = doc_data["meta"]
        content = doc_data["content"]
        
        precomputed = rels_map.get(s_doc, [])
        
        key = normalize_doc_key(s_doc)
        if key: meta_by_docnum_lookup[key] = meta
        
        doc_chunks = chunker.process_document(
            content=content,
            metadata=meta,
            global_doc_lookup=global_doc_lookup,
            precomputed_rels=precomputed
        )
        all_chunks.extend(doc_chunks)
        processed_ids.add(doc_id)

        # Discovery: Phát hiện các văn bản tham chiếu
        for chunk in doc_chunks:
            neo4j_meta = chunk.get("neo4j_metadata", chunk.get("metadata", {}))
            for r in neo4j_meta.get("ontology_relations", []):
                t_key = normalize_doc_key(r.get("target_doc", ""))
                t_id = meta_docnum_to_id.get(t_key)
                if t_id and t_id not in processed_ids and t_id not in target_ids:
                    discovered_depth1_ids.add(t_id)

    print(f"  Đã hoàn thành mẻ {i//BATCH_SIZE + 1}. Tổng processed: {len(processed_ids)}/{len(target_ids)}. Target docs mới: {len(discovered_depth1_ids)}")

import random
# Giới hạn số lượng target docs (lấy ngẫu nhiên)
if len(discovered_depth1_ids) > MAX_RECURSIVE_DOCS:
    print(f"  ⚠️ Phát hiện {len(discovered_depth1_ids)} target docs, giới hạn còn {MAX_RECURSIVE_DOCS}.")
    # Chuyển set thành list và dùng random.sample để lấy ngẫu nhiên không lặp lại
    discovered_depth1_ids = set(random.sample(list(discovered_depth1_ids), MAX_RECURSIVE_DOCS))

print(f"\n✅ PHA 1 hoàn tất: {len(processed_ids)} văn bản gốc → {len(all_chunks)} chunks.")
print(f"   Phát hiện {len(discovered_depth1_ids)} văn bản tham chiếu cần nạp thêm.")


In [ ]:
# ═══════════════════════════════════════════════════
# PHA 2: Chunk các văn bản THAM CHIẾU (depth=1) — KHÔNG đệ quy thêm
# ═══════════════════════════════════════════════════
print("\n" + "═" * 60)
print("PHA 2: Chunking văn bản THAM CHIẾU (depth=1) (BATCH 10)")
print("═" * 60)

depth1_count = 0
disco_ids_list = list(discovered_depth1_ids)
for i in range(0, len(disco_ids_list), BATCH_SIZE):
    batch_ids = disco_ids_list[i : i + BATCH_SIZE]
    batch_docs = []
    batch_meta_map = {}
    
    for doc_id in batch_ids:
        if doc_id in processed_ids: continue
        
        idx_content = content_id_to_idx.get(doc_id)
        idx_meta = meta_id_to_idx.get(doc_id)
        if idx_content is None or idx_meta is None:
            processed_ids.add(doc_id)
            continue
        
        content = ds_content_all[idx_content].get("content", "")
        meta = all_meta_records[idx_meta]
        if not content: 
            processed_ids.add(doc_id)
            continue
            
        doc_num = str(meta.get("document_number", ""))
        batch_docs.append({"source_doc": doc_num, "content": content})
        batch_meta_map[doc_num] = {"id": doc_id, "meta": meta, "content": content}

    if not batch_docs: continue

    # BƯỚC 1: TRÍCH XUẤT RELATION THEO MẺ 10 (Cell 3 & 4)
    print(f"  [BATCH] Đang trích xuất quan hệ cho mẻ {len(batch_docs)} văn bản...")
    
    # Bọc Try-Catch để đảm bảo pipeline không bao giờ chết
    try:
        rels_map = rel.extract_ontology_relationships_batch(batch_docs, global_doc_lookup=global_doc_lookup)
    except Exception as e:
        print(f"  ⚠️ LỖI BATCH: Bỏ qua trích xuất quan hệ mẻ này do lỗi LLM - {e}")
        rels_map = {} # Dự phòng rỗng để vẫn chunk được văn bản gốc
    
    for doc_info in batch_docs:
        s_doc = doc_info["source_doc"]
        doc_data = batch_meta_map[s_doc]
        doc_id = doc_data["id"]
        meta = doc_data["meta"]
        content = doc_data["content"]
        
        precomputed = rels_map.get(s_doc, [])
        
        key = normalize_doc_key(s_doc)
        if key: meta_by_docnum_lookup[key] = meta
        
        doc_chunks = chunker.process_document(
            content=content,
            metadata=meta,
            global_doc_lookup=global_doc_lookup,
            precomputed_rels=precomputed
        )
        all_chunks.extend(doc_chunks)
        processed_ids.add(doc_id)
        depth1_count += 1

    print(f"  Đã hoàn thành mẻ tham chiếu {i//BATCH_SIZE + 1}. Tổng xử lý: {depth1_count}/{len(disco_ids_list)}")

print(f"\n✅ PHA 2 hoàn tất: Thêm {depth1_count} văn bản tham chiếu.")
print(f"🎯 TỔNG KẾT: {len(processed_ids)} văn bản ({len(target_ids)} gốc + {depth1_count} tham chiếu) → {len(all_chunks)} chunks.")


In [ ]:
import uuid
from backend.retrieval.embedder import embedder
from backend.retrieval.vector_db import client as qdrant, ensure_collection
from backend.retrieval.graph_db import get_neo4j_driver, build_neo4j
from qdrant_client.models import PointStruct
from backend.config import settings

collection_name = "legal_hybrid_rag_docs"

# 0. Flush/Clear Database Cũ
print("-> Đang xóa hoàn toàn Database cũ (Flush DB)...")
if qdrant.collection_exists(collection_name):
    qdrant.delete_collection(collection_name=collection_name)
    print("   ✅ Đã xóa Qdrant Collection.")

driver = get_neo4j_driver()
if driver:
    with driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")
    print("   ✅ Đã xóa tất cả Node và Relation trong Neo4j.")
    driver.close()

# Đảm bảo Collection tồn tại trước khi thao tác
ensure_collection(qdrant, collection_name=collection_name)

# 1. Tính toán Dense và Sparse Vectors (Batching để tránh Timeout)
print("-> Bắt đầu trích xuất Vectors (Batching = 50)...")
texts_to_embed = [c.get("text_to_embed", c.get("chunk_text", "")) for c in all_chunks]

dense_vectors = []
sparse_vectors = []
batch_size = 50

for i in range(0, len(texts_to_embed), batch_size):
    batch_texts = texts_to_embed[i:i + batch_size]
    # encode_dense
    dense_batch = embedder.encode_dense(batch_texts)
    dense_vectors.extend(dense_batch)
    
    # encode_sparse_documents (để sinh SparseVector placeholder)
    sparse_batch = embedder.encode_sparse_documents(batch_texts)
    sparse_vectors.extend(sparse_batch)
    
    print(f"   Đã nhúng {min(i + batch_size, len(texts_to_embed))}/{len(texts_to_embed)} chunks...")

# 2. Đẩy Qdrant (Batching để chống Sập/OOM Docker)
print("-> Đang đẩy vào Qdrant (Batching = 200)...")
points = []
for idx, chunk in enumerate(all_chunks):
    vec_payload = {
        "dense": dense_vectors[idx],
        "sparse": sparse_vectors[idx]
    }
    q_payload = chunk.get("qdrant_metadata", chunk.get("metadata", {}))
    points.append(PointStruct(
        id=str(uuid.uuid5(uuid.NAMESPACE_DNS, chunk["chunk_id"])),
        vector=vec_payload,
        payload=q_payload
    ))

upsert_batch_size = 200
for i in range(0, len(points), upsert_batch_size):
    batch_points = points[i:i + upsert_batch_size]
    qdrant.upsert(collection_name=collection_name, points=batch_points)
    print(f"   Đã đẩy Qdrant {min(i + upsert_batch_size, len(points))}/{len(points)} chunks...")

print(f"✅ Đã tải thành công lên Local Qdrant (Collection: {collection_name}).")

# 3. Đẩy Graph Neo4j
print("-> Đang xây dựng Đồ thị tri thức (Neo4j)...")
driver = get_neo4j_driver()
if driver:
    build_neo4j(driver, all_chunks, meta_by_docnum_lookup=meta_by_docnum_lookup)
    driver.close()
    print("✅ Cập nhật Graph Neo4j hoàn thành.")
else:
    print("❌ Lỗi kết nối Neo4j.")

In [ ]:
# import os
# # Đảm bảo bạn đang ở thư mục gốc của dự án
# os.chdir("d:/iCOMM/Legal-RAG")
# # Chạy toàn bộ test suite
# !python tests/run_all_tests.py